# Data Cleaning

This notebook loads the raw historical NFL player-season dataset, validates the data, calculates full-PPR fantasy points, and prepares a clean dataset for exploratory analysis and feature engineering.

In [1]:
import polars as pl

In [2]:
df  = pl.read_csv("../data/raw/player_stats_2016_2025.csv")

df.head()

player_id,player_display_name,position,season,recent_team,games,attempts,passing_yards,passing_tds,passing_interceptions,passing_2pt_conversions,carries,rushing_yards,rushing_tds,rushing_2pt_conversions,targets,receptions,receiving_yards,receiving_tds,receiving_2pt_conversions,fumbles_lost_total,sack_fumbles_lost,rushing_fumbles_lost,receiving_fumbles_lost,special_teams_tds,target_share,air_yards_share,wopr,fantasy_points,fantasy_points_ppr,calculated_fantasy_points_ppr
str,str,str,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64
"""00-0019596""","""Tom Brady""","""QB""",2016,"""NE""",12,432,3554,28,2,1,28,64,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,258.56,258.56,258.56
"""00-0020337""","""Steve Smith""","""WR""",2016,"""BAL""",14,0,0,0,0,0,0,0,0,0,101,70,799,5,2,0,0,0,0,0,0.15188,0.173146,0.349021,113.9,183.9,183.9
"""00-0020531""","""Drew Brees""","""QB""",2016,"""NO""",16,673,5208,37,15,0,23,20,2,0,0,0,0,0,0,4,4,0,0,0,0.0,0.0,0.0,332.32,332.32,332.32
"""00-0020679""","""Shaun Hill""","""QB""",2016,"""MIN""",3,35,242,0,0,0,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,10.18,10.18,10.18
"""00-0021206""","""Josh McCown""","""QB""",2016,"""CLE""",5,165,1100,6,6,1,7,21,0,0,0,0,0,0,0,4,4,0,0,0,0.0,0.0,0.0,52.1,52.1,52.1


In [3]:
print(df.shape)
df.group_by("season").len().sort("season")

(5874, 31)


season,len
i64,u32
2016,557
2017,553
2018,577
2019,572
2020,602
2021,633
2022,608
2023,576
2024,588


In [4]:
# Checking for low participation players

df.filter(
    pl.col("games") <= 2
).select([
    "player_display_name",
    "position",
    "season",
    "recent_team",
    "games",
    "fantasy_points_ppr"
]).sort(
    ["season", "position", "fantasy_points_ppr"],
    descending=[False, False, True]
)

player_display_name,position,season,recent_team,games,fantasy_points_ppr
str,str,i64,str,i64,f64
"""Nick Foles""","""QB""",2016,"""KC""",2,28.0
"""Charlie Whitehurst""","""QB""",2016,"""CLE""",1,9.38
"""Mike Glennon""","""QB""",2016,"""TB""",1,9.0
"""Scott Tolzien""","""QB""",2016,"""IND""",2,8.94
"""Geno Smith""","""QB""",2016,"""NYJ""",2,7.94
…,…,…,…,…,…
"""DJ Turner""","""WR""",2025,"""LV""",1,0.0
"""Mecole Hardman""","""WR""",2025,"""BUF""",2,0.0
"""Kaden Davis""","""WR""",2025,"""CLE""",1,0.0


In [5]:
df.filter(
    pl.col("games") <= 2
).height

827

In [6]:
# Recalculating and locking in my PPR target

df = df.with_columns(
    (
        pl.col("passing_yards") / 25
        + pl.col("passing_tds") * 4
        - pl.col("passing_interceptions") * 2

        + pl.col("rushing_yards") / 10
        + pl.col("rushing_tds") * 6

        + pl.col("receiving_yards") / 10
        + pl.col("receptions")
        + pl.col("receiving_tds") * 6

        + pl.col("passing_2pt_conversions") * 2
        + pl.col("rushing_2pt_conversions") * 2
        + pl.col("receiving_2pt_conversions") * 2

        + pl.col("special_teams_tds") * 6

        - (
            pl.col("sack_fumbles_lost")
            + pl.col("rushing_fumbles_lost")
            + pl.col("receiving_fumbles_lost")
        ) * 2
    ).alias("fantasy_points_ppr_calc")
)

In [7]:
df = df.with_columns(
    (
        pl.col("fantasy_points_ppr_calc")
        - pl.col("fantasy_points_ppr")
    ).alias("fantasy_points_difference")
)

df.select(
    pl.col("fantasy_points_difference").abs().mean().alias("mean_abs_difference"),
    pl.col("fantasy_points_difference").abs().max().alias("max_abs_difference")
)

mean_abs_difference,max_abs_difference
f64,f64
1.0445e-15,5.6843e-14


In [8]:
# Saving cleaned dataset

clean_path  = "../data/processed/player_stats_clean_2016_2025"

df.write_csv(clean_path)

In [9]:
import os

os.path.exists(clean_path)

True